# Сравнение `agr_id` из Excel и витрины

Тетрадка сравнивает множества `agr_id`:
- из Excel-отчета,
- из витрины `sandbox_ai.stg__chesnov_aef__sa_acquiring_datamart` за выбранный месяц.

Результат:
- `intersect_cnt`
- `only_excel_cnt`
- `only_datamart_cnt`

In [ ]:
import pandas as pd

excel_path = "/home/jovyan/documents/Equaring/Проверки/02_февраль.xlsx"  # путь к Excel
agr_col = "agr_id"          # имя колонки agr_id в Excel
month_start = "2026-02-01"  # YYYY-MM-01

In [ ]:
# 1) agr_id из Excel
df = pd.read_excel(excel_path)

excel_agr = (
    df[agr_col]
    .astype(str)
    .str.strip()
    .replace({"": None, "nan": None, "None": None})
    .dropna()
    .str.replace(r"\\.0$", "", regex=True)
    .drop_duplicates()
)

print("excel_rows =", len(df))
print("excel_unique_agr =", len(excel_agr))
excel_agr.head()

In [ ]:
# 2) agr_id из витрины за месяц
with imp:
    dm = imp.fetch(f"""
        select distinct cast(agr_id as string) as agr_id
        from sandbox_ai.stg__chesnov_aef__sa_acquiring_datamart
        where trx_month = cast('{month_start}' as date)
          and agr_id is not null
    """)

dm_agr = (
    dm["agr_id"]
    .astype(str)
    .str.strip()
    .replace({"": None, "nan": None, "None": None})
    .dropna()
    .str.replace(r"\\.0$", "", regex=True)
    .drop_duplicates()
)

print("datamart_unique_agr =", len(dm_agr))
dm_agr.head()

In [ ]:
# 3) Сравнение множеств
set_excel = set(excel_agr.tolist())
set_dm = set(dm_agr.tolist())

intersect = set_excel & set_dm
only_excel = set_excel - set_dm
only_dm = set_dm - set_excel

summary = pd.DataFrame([
    {"metric": "excel_unique_agr", "value": len(set_excel)},
    {"metric": "datamart_unique_agr", "value": len(set_dm)},
    {"metric": "intersect_cnt", "value": len(intersect)},
    {"metric": "only_excel_cnt", "value": len(only_excel)},
    {"metric": "only_datamart_cnt", "value": len(only_dm)},
])
summary

In [ ]:
# 4) Примеры расхождений
only_excel_df = pd.DataFrame({"agr_id": sorted(list(only_excel))})
only_datamart_df = pd.DataFrame({"agr_id": sorted(list(only_dm))})

print("Examples: only in Excel")
display(only_excel_df.head(50))

print("Examples: only in Datamart")
display(only_datamart_df.head(50))

In [ ]:
# 5) Выбираем agr_id, который есть и в Excel, и в Datamart
common_agr = sorted(list(intersect))
print("common agr count:", len(common_agr))

# Можно заменить на конкретный agr_id из пересечения
# target_agr = "123456789"
target_agr = common_agr[0] if common_agr else None
print("target_agr =", target_agr)


In [ ]:
# 6) Строки Excel по выбранному agr_id
excel_cmp = df.copy()
excel_cmp[agr_col] = (
    excel_cmp[agr_col]
    .astype(str)
    .str.strip()
    .str.replace(r"\\.0$", "", regex=True)
)

if target_agr is None:
    excel_one = pd.DataFrame()
else:
    excel_one = excel_cmp[excel_cmp[agr_col] == str(target_agr)].copy()

print("excel rows for target agr:", len(excel_one))
display(excel_one.head(20))


In [ ]:
# 7) Строки Datamart по выбранному agr_id за месяц
if target_agr is None:
    dm_one = pd.DataFrame()
else:
    with imp:
        dm_one = imp.fetch(f"""
            select *
            from sandbox_ai.stg__chesnov_aef__sa_acquiring_datamart
            where trx_month = cast('{month_start}' as date)
              and cast(agr_id as string) = '{target_agr}'
        """)

print("datamart rows for target agr:", len(dm_one))
display(dm_one.head(20))


In [ ]:
# 8) Сводное сравнение ключевых полей Excel vs Datamart
# ВАЖНО: замените названия Excel-колонок справа на фактические имена в вашем файле.
excel_cols = {
    "inn": "ИНН",
    "tariff": "Тариф",
    "vsp_code": "Код ВСП",
    "vsp_name": "ВСП договора",
    "sum_trx_amt": "Сумма операций",
    "cnt_trx": "Кол-во операций",
}

# Названия полей Datamart (проверьте и при необходимости поправьте под вашу схему).
dm_cols = {
    "inn": "inn",
    "tariff": "tariff_name",
    "vsp_code": "code_depart",
    "vsp_name": "name_depart",
    "sum_trx_amt": "sum_trx_amt",
    "cnt_trx": "cnt_trx",
}

excel_row = excel_one.iloc[0] if len(excel_one) > 0 else pd.Series(dtype="object")

if len(dm_one) > 0:
    dm_month = {}
    for metric_key, dm_col in dm_cols.items():
        if dm_col not in dm_one.columns:
            dm_month[metric_key] = None
            continue

        # Метрики в Datamart часто на дневной зернистости, поэтому сводим за месяц.
        if metric_key in ["sum_trx_amt", "cnt_trx"]:
            dm_month[metric_key] = pd.to_numeric(dm_one[dm_col], errors="coerce").fillna(0).sum()
        else:
            series = dm_one[dm_col].dropna().astype(str).str.strip()
            dm_month[metric_key] = series.iloc[0] if len(series) > 0 else None
else:
    dm_month = {k: None for k in dm_cols.keys()}

rows = []
for metric_key, excel_col in excel_cols.items():
    excel_val = excel_row.get(excel_col, None) if len(excel_one) > 0 else None
    dm_val = dm_month.get(metric_key, None)

    rows.append({
        "metric": metric_key,
        "excel_column": excel_col,
        "datamart_column": dm_cols.get(metric_key),
        "excel_value": excel_val,
        "datamart_value": dm_val,
        "is_equal_as_str": str(excel_val) == str(dm_val),
    })

cmp_df = pd.DataFrame(rows)
display(cmp_df)


In [ ]:
# 9) (Опционально) Быстрая проверка: какие из ожидаемых полей реально есть в Datamart
expected_dm_cols = sorted(set(dm_cols.values()))
actual_dm_cols = set(dm_one.columns.tolist()) if len(dm_one) > 0 else set()

availability = pd.DataFrame({
    "datamart_column": expected_dm_cols,
    "exists_in_result": [c in actual_dm_cols for c in expected_dm_cols]
})

display(availability)
